In [2]:
# CELL 2: Environment Setup & AMD Hardware Acceleration Verification

import os
import sys  
import subprocess

def setup_environment():
    print("[+] Checking and Installing Core System Dependencies...")
    
    # Excluded built-in modules like sqlite3 from pip installation list
    dependencies = [
        "langchain",
        "langchain-community",
        "langchain-core",
        "langchain-ollama",
        "chromadb",
        "pydantic",
        "google-api-python-client",
        "google-auth-httplib2",
        "google-auth-oauthlib",
        "gradio",
        "pandas",
        "numpy"
    ]
    
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + dependencies
    subprocess.check_call(cmd)
    print("[✓] All external Python packages successfully installed.")

def verify_amd_hardware():
    print("\n[+] Verifying AMD Acceleration Environment (ROCm / Ryzen AI)...")
    
    try:
        import torch
        if torch.cuda.is_available():
            device_name = torch.cuda.get_device_name(0)
            print(f"    [Device Detected]: {device_name}")
            if "AMD" in device_name or "Radeon" in device_name or "ROCm" in torch.__version__:
                print("    [Status]: AMD ROCm GPU Acceleration ACTIVE.")
            else:
                print("    [Status]: GPU Detected.")
        else:
            print("    [Status]: Running on AMD Ryzen AI / CPU Acceleration Mode.")
        
        torch_ver = torch.__version__
    except ImportError:
        torch_ver = "Not Installed"
        print("    [Status]: Running on Native CPU Execution Mode.")

    hip_visible = os.environ.get("HIP_VISIBLE_DEVICES", "Not Set")
    print(f"    [HIP_VISIBLE_DEVICES]: {hip_visible}")
    
    return {
        "torch_version": torch_ver,
        "python_version": sys.version.split()[0]
    }

# Execute Setup
setup_environment()
hw_status = verify_amd_hardware()
print("\n[✓] Environment Initialization Complete.")

[+] Checking and Installing Core System Dependencies...
[✓] All external Python packages successfully installed.

[+] Verifying AMD Acceleration Environment (ROCm / Ryzen AI)...
    [Status]: Running on Native CPU Execution Mode.
    [HIP_VISIBLE_DEVICES]: Not Set

[✓] Environment Initialization Complete.


In [2]:
##Cell - 4
import os
import json
import sqlite3
import warnings
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field

# Suppress deprecation and library warnings for clean notebook output
warnings.filterwarnings("ignore")

from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Initialize Directory Structure
DATA_DIR = "./agent_data"
DB_PATH = os.path.join(DATA_DIR, "agent_memory.db")
CHROMA_DIR = os.path.join(DATA_DIR, "chroma_db")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHROMA_DIR, exist_ok=True)

# Define Model Configurations
MODEL_NAME = "llama3"
EMBEDDING_MODEL = "nomic-embed-text"

print(f"[+] Initializing Ollama Local LLM Engine ({MODEL_NAME})...")
try:
    llm = OllamaLLM(model=MODEL_NAME, temperature=0.1)
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
    print("[✓] Ollama Engine & Embeddings Connected Successfully.")
except Exception as e:
    print(f"[!] Warning connecting to Ollama: {e}")

# Initialize Chroma Vector Store for Long-Term Memory
vector_db = Chroma(
    collection_name="agent_long_term_memory",
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)
print("[✓] Persistent ChromaDB Vector Memory Initialized.")

# Initialize SQLite Relational Database for Structured Logs
def init_sqlite_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS audit_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
            source TEXT,
            sender TEXT,
            subject TEXT,
            priority TEXT,
            security_status TEXT,
            decision TEXT,
            reasoning TEXT,
            raw_payload TEXT
        )
    ''')
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS user_preferences (
            key TEXT PRIMARY KEY,
            value TEXT
        )
    ''')
    
    conn.commit()
    conn.close()
    print("[✓] SQLite Audit & Preference Database Initialized.")

init_sqlite_db()

[+] Initializing Ollama Local LLM Engine (llama3)...
[✓] Ollama Engine & Embeddings Connected Successfully.
[✓] Persistent ChromaDB Vector Memory Initialized.
[✓] SQLite Audit & Preference Database Initialized.


In [5]:
#==============================================================================
# CELL 6: Security Agent & Notification Aggregator Implementation
# ==============================================================================

import re
from urllib.parse import urlparse

class SecurityAgent:
    """Agent responsible for inspecting messages for phishing, fake domains, and social engineering."""
    
    SUSPICIOUS_KEYWORDS = ["urgent wire transfer", "verify password immediately", "account suspended", "gift card", "crypto deposit"]
    KNOWN_DOMAINS = ["company.com", "gmail.com", "github.com", "slack.com", "jira.atlassian.net"]

    @staticmethod
    def analyze_message(sender: str, body: str, links: List[str] = []) -> Dict[str, Any]:
        risk_score = 0
        reasons = []

        # 1. Sender Domain Verification
        sender_domain = sender.split("@")[-1] if "@" in sender else ""
        if sender_domain and sender_domain not in KNOWN_DOMAINS:
            risk_score += 25
            reasons.append(f"Unverified external domain: @{sender_domain}")

        # 2. Keyword / Social Engineering Threat Scanning
        for kw in SecurityAgent.SUSPICIOUS_KEYWORDS:
            if kw.lower() in body.lower():
                risk_score += 35
                reasons.append(f"Suspicious social engineering phrase detected: '{kw}'")

        # 3. Malicious Link Analysis
        for link in links:
            parsed = urlparse(link)
            if re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', parsed.netloc):
                risk_score += 40
                reasons.append(f"Direct IP raw URL detected: {link}")

        # Verdict
        if risk_score >= 60:
            status = "CRITICAL_PHISHING_RISK"
        elif risk_score >= 25:
            status = "SUSPICIOUS"
        else:
            status = "SAFE"

        return {
            "status": status,
            "risk_score": risk_score,
            "threats_detected": reasons,
            "is_safe": status == "SAFE"
        }

class NotificationAggregationAgent:
    """Aggregates and standardizes messages from multi-platform sources into a single schema."""
    
    @staticmethod
    def normalize_event(source: str, sender: str, title: str, content: str, timestamp: str) -> Dict[str, Any]:
        return {
            "source": source,
            "sender": sender,
            "title": title,
            "content": content,
            "timestamp": timestamp,
            "id": f"{source}_{hash(content)}"
        }

print("[✓] Security Agent and Multi-Platform Aggregator loaded.")

[✓] Security Agent and Multi-Platform Aggregator loaded.


In [6]:
# ==============================================================================
# CELL 6: Security Agent & Notification Aggregator Implementation
# ==============================================================================

import re
from urllib.parse import urlparse

class SecurityAgent:
    """Agent responsible for inspecting messages for phishing, fake domains, and social engineering."""
    
    SUSPICIOUS_KEYWORDS = ["urgent wire transfer", "verify password immediately", "account suspended", "gift card", "crypto deposit"]
    KNOWN_DOMAINS = ["company.com", "gmail.com", "github.com", "slack.com", "jira.atlassian.net"]

    @staticmethod
    def analyze_message(sender: str, body: str, links: List[str] = []) -> Dict[str, Any]:
        risk_score = 0
        reasons = []

        # 1. Sender Domain Verification
        sender_domain = sender.split("@")[-1] if "@" in sender else ""
        if sender_domain and sender_domain not in KNOWN_DOMAINS:
            risk_score += 25
            reasons.append(f"Unverified external domain: @{sender_domain}")

        # 2. Keyword / Social Engineering Threat Scanning
        for kw in SecurityAgent.SUSPICIOUS_KEYWORDS:
            if kw.lower() in body.lower():
                risk_score += 35
                reasons.append(f"Suspicious social engineering phrase detected: '{kw}'")

        # 3. Malicious Link Analysis
        for link in links:
            parsed = urlparse(link)
            if re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', parsed.netloc):
                risk_score += 40
                reasons.append(f"Direct IP raw URL detected: {link}")

        # Verdict
        if risk_score >= 60:
            status = "CRITICAL_PHISHING_RISK"
        elif risk_score >= 25:
            status = "SUSPICIOUS"
        else:
            status = "SAFE"

        return {
            "status": status,
            "risk_score": risk_score,
            "threats_detected": reasons,
            "is_safe": status == "SAFE"
        }

class NotificationAggregationAgent:
    """Aggregates and standardizes messages from multi-platform sources into a single schema."""
    
    @staticmethod
    def normalize_event(source: str, sender: str, title: str, content: str, timestamp: str) -> Dict[str, Any]:
        return {
            "source": source,
            "sender": sender,
            "title": title,
            "content": content,
            "timestamp": timestamp,
            "id": f"{source}_{hash(content)}"
        }

print("[✓] Security Agent and Multi-Platform Aggregator loaded.")

[✓] Security Agent and Multi-Platform Aggregator loaded.


In [7]:
# ==============================================================================
# CELL 8: Context, Prioritization, and Long-Term Memory Agents
# ==============================================================================

class LongTermMemoryAgent:
    """Manages semantic search over past emails and user preferences using ChromaDB."""
    
    def __init__(self, vector_store):
        self.db = vector_store

    def store_interaction(self, sender: str, summary: str, user_decision: str):
        text_data = f"Sender: {sender} | Summary: {summary} | Action Taken: {user_decision}"
        doc = Document(page_content=text_data, metadata={"sender": sender, "action": user_decision})
        self.db.add_documents([doc])

    def query_context(self, sender: str, query_text: str, k: int = 2) -> List[str]:
        results = self.db.similarity_search(f"{sender} {query_text}", k=k)
        return [res.page_content for res in results]

class PrioritizationAgent:
    """Ranks communications using contextual reasoning rather than basic keyword rules."""
    
    def __init__(self, llm_engine):
        self.llm = llm_engine

    def evaluate_priority(self, sender: str, subject: str, content: str, memory_context: str) -> Dict[str, Any]:
        prompt = f"""
        Analyze the priority of this message based on sender importance, urgency, and context.
        Sender: {sender}
        Subject: {subject}
        Content: {content}
        Past Memory Context: {memory_context}
        
        Respond ONLY in raw JSON format with keys:
        - priority_level: (URGENT, HIGH, MEDIUM, LOW)
        - score: (1 to 100)
        - reasoning: (1 sentence explaining why)
        """
        try:
            response = self.llm.invoke(prompt)
            # Find JSON payload
            json_match = re.search(r'\{.*\}', response, re.DOTALL)
            if json_match:
                return json.loads(json_match.group(0))
        except Exception:
            pass

        return {"priority_level": "MEDIUM", "score": 50, "reasoning": "Fallback default evaluation."}

print("[✓] Memory & Prioritization Agents loaded.")

[✓] Memory & Prioritization Agents loaded.


In [8]:
# ==============================================================================
# CELL 9: Smart Reply, Approval Decision, Summarization & Security Orchestrator
# ==============================================================================

class SmartReplyAgent:
    """Generates personalized, style-matched responses using conversation history."""
    def __init__(self, llm_engine):
        self.llm = llm_engine

    def generate_reply(self, sender: str, content: str, memory_context: List[str]) -> str:
        past_interactions = "\n".join(memory_context) if memory_context else "No prior history."
        prompt = f"""
        Generate a professional, concise, and natural reply to the following message.
        Match a helpful, direct tone.

        Sender: {sender}
        Message: {content}
        Past Context with Sender:
        {past_interactions}

        Draft Response:
        """
        return self.llm.invoke(prompt).strip()

class ApprovalDecisionAgent:
    """Determines whether a message can be Auto-Replied or requires Manual User Approval."""
    @staticmethod
    def evaluate(priority: str, risk_status: str, confidence_score: float) -> Dict[str, Any]:
        if risk_status != "SAFE":
            return {"action": "NEVER_AUTO_REPLY", "reason": f"Security risk detected ({risk_status})."}
        if priority == "URGENT":
            return {"action": "REQUIRES_APPROVAL", "reason": "High urgency requires human review."}
        if confidence_score >= 0.85:
            return {"action": "AUTO_REPLY", "reason": "High confidence score and clean security scan."}
        return {"action": "REQUIRES_APPROVAL", "reason": "Moderate confidence score."}

class ThreadSummarizationAgent:
    """Extracts summary, key action items, deadlines, and meetings."""
    def __init__(self, llm_engine):
        self.llm = llm_engine

    def process(self, content: str) -> Dict[str, Any]:
        prompt = f"""
        Extract structured information from this text. Respond strictly in JSON format.
        Text: {content}

        Required Keys:
        - summary: (1 sentence summary)
        - action_items: (list of tasks)
        - deadlines: (list of deadlines if any)
        - meetings: (list of scheduled meetings if any)
        """
        try:
            res = self.llm.invoke(prompt)
            match = re.search(r'\{.*\}', res, re.DOTALL)
            if match:
                return json.loads(match.group(0))
        except Exception:
            pass
        return {
            "summary": content[:100] + "...",
            "action_items": [],
            "deadlines": [],
            "meetings": []
        }

class ExplainabilityAgent:
    """Formats decision traces for complete auditability."""
    @staticmethod
    def generate_trace(msg_id: str, priority_res: dict, sec_res: dict, approval_res: dict) -> str:
        return f"""
        [TRACE - {msg_id}]
        ├── Priority : {priority_res.get('priority_level')} (Score: {priority_res.get('score')})
        │   └── Reason: {priority_res.get('reasoning')}
        ├── Security : {sec_res.get('status')} (Risk Score: {sec_res.get('risk_score')})
        │   └── Threat Analysis: {sec_res.get('threats_detected')}
        └── Decision : {approval_res.get('action')}
            └── Policy Reason: {approval_res.get('reason')}
        """

print("[✓] Smart Reply, Approval, Summarization & Explainability Agents initialized.")

[✓] Smart Reply, Approval, Summarization & Explainability Agents initialized.


In [9]:
# ==============================================================================
# CELL 10: Master Autonomous Loop Engine & Gmail API Tool Integrations
# ==============================================================================

class MasterAutonomousAgentLoop:
    """Core Observe -> Reason -> Plan -> Select Tool -> Execute -> Verify -> Learn Engine."""
    
    def __init__(self, llm_engine, vector_store):
        self.llm = llm_engine
        self.memory = LongTermMemoryAgent(vector_store)
        self.sec_agent = SecurityAgent()
        self.prioritizer = PrioritizationAgent(llm_engine)
        self.reply_agent = SmartReplyAgent(llm_engine)
        self.summarizer = ThreadSummarizationAgent(llm_engine)

    def process_incoming_communication(self, source: str, sender: str, subject: str, content: str) -> Dict[str, Any]:
        # Step 1: OBSERVE & AGGREGATE
        event = NotificationAggregationAgent.normalize_event(source, sender, subject, content, "2026-07-21")
        
        # Step 2: SECURITY SCAN
        sec_result = self.sec_agent.analyze_message(sender, content)
        
        # Step 3: MEMORY RETRIEVAL
        past_context = self.memory.query_context(sender, content)
        
        # Step 4: REASON & PRIORITIZE
        prio_result = self.prioritizer.evaluate_priority(sender, subject, content, str(past_context))
        
        # Step 5: SUMMARIZE & EXTRACT TASKS
        summary_result = self.summarizer.process(content)
        
        # Step 6: GENERATE DRAFT REPLY
        draft_reply = self.reply_agent.generate_reply(sender, content, past_context)
        
        # Step 7: APPROVAL & AUTONOMY DECISION
        approval_result = ApprovalDecisionAgent.evaluate(
            prio_result.get("priority_level"),
            sec_result.get("status"),
            confidence_score=0.90 if sec_result.get("is_safe") else 0.20
        )
        
        # Step 8: EXPLAINABILITY TRACE
        trace = ExplainabilityAgent.generate_trace(event["id"], prio_result, sec_result, approval_result)
        
        # Step 9: EXECUTE & UPDATE MEMORY
        if approval_result["action"] == "AUTO_REPLY":
            self.memory.store_interaction(sender, summary_result["summary"], "AUTONOMOUS_REPLY_SENT")
        else:
            self.memory.store_interaction(sender, summary_result["summary"], "HELD_FOR_APPROVAL")

        # Step 10: AUDIT LOGGING
        self._log_to_sqlite(source, sender, subject, prio_result.get("priority_level"), sec_result.get("status"), approval_result.get("action"), trace)

        return {
            "event": event,
            "security": sec_result,
            "priority": prio_result,
            "summary": summary_result,
            "draft_reply": draft_reply,
            "decision": approval_result,
            "trace": trace
        }

    def _log_to_sqlite(self, source, sender, subject, priority, security, decision, reasoning):
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute('''
            INSERT INTO audit_logs (source, sender, subject, priority, security_status, decision, reasoning, raw_payload)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (source, sender, subject, priority, security, decision, reasoning, ""))
        conn.commit()
        conn.close()

# Initialize Engine
agent_loop = MasterAutonomousAgentLoop(llm, vector_db)
print("[✓] Master Autonomous Agent Loop Initialized.")

[✓] Master Autonomous Agent Loop Initialized.


In [10]:
# ==============================================================================
# CELL 11: Multi-Platform Autonomous Simulation & Live Test Execution (FULLY SELF-CONTAINED)
# ==============================================================================

import os
import json
import sqlite3
import re
from typing import List, Dict, Any
from urllib.parse import urlparse

# Ensure DATA_DIR and DB_PATH are set
DATA_DIR = "./agent_data"
DB_PATH = os.path.join(DATA_DIR, "agent_memory.db")
os.makedirs(DATA_DIR, exist_ok=True)

# ------------------------------------------------------------------------------
# Fallback Definitions for All Required Agents (Ensures 100% Independent Execution)
# ------------------------------------------------------------------------------

class SecurityAgent:
    SUSPICIOUS_KEYWORDS = ["urgent wire transfer", "verify password immediately", "account suspended", "gift card", "crypto deposit"]
    KNOWN_DOMAINS = ["company.com", "gmail.com", "github.com", "slack.com", "jira.atlassian.net", "amd.com", "techexecs.com"]

    @classmethod
    def analyze_message(cls, sender: str, body: str, links: List[str] = []) -> Dict[str, Any]:
        risk_score = 0
        reasons = []

        sender_domain = sender.split("@")[-1] if "@" in sender else ""
        if sender_domain and sender_domain not in cls.KNOWN_DOMAINS:
            risk_score += 25
            reasons.append(f"Unverified external domain: @{sender_domain}")

        for kw in cls.SUSPICIOUS_KEYWORDS:
            if kw.lower() in body.lower():
                risk_score += 35
                reasons.append(f"Suspicious social engineering phrase detected: '{kw}'")

        for link in links:
            parsed = urlparse(link)
            if re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', parsed.netloc):
                risk_score += 40
                reasons.append(f"Direct IP raw URL detected: {link}")

        status = "CRITICAL_PHISHING_RISK" if risk_score >= 60 else ("SUSPICIOUS" if risk_score >= 25 else "SAFE")

        return {
            "status": status,
            "risk_score": risk_score,
            "threats_detected": reasons,
            "is_safe": status == "SAFE"
        }

class NotificationAggregationAgent:
    @staticmethod
    def normalize_event(source: str, sender: str, title: str, content: str, timestamp: str) -> Dict[str, Any]:
        return {
            "source": source,
            "sender": sender,
            "title": title,
            "content": content,
            "timestamp": timestamp,
            "id": f"{source}_{abs(hash(content))}"
        }

class LongTermMemoryAgent:
    def __init__(self, vector_store=None):
        self.db = vector_store

    def store_interaction(self, sender: str, summary: str, user_decision: str):
        if self.db:
            from langchain_core.documents import Document
            text_data = f"Sender: {sender} | Summary: {summary} | Action Taken: {user_decision}"
            doc = Document(page_content=text_data, metadata={"sender": sender, "action": user_decision})
            self.db.add_documents([doc])

    def query_context(self, sender: str, query_text: str, k: int = 2) -> List[str]:
        if self.db:
            try:
                results = self.db.similarity_search(f"{sender} {query_text}", k=k)
                return [res.page_content for res in results]
            except Exception:
                pass
        return [f"Prior interaction logged with {sender}"]

class PrioritizationAgent:
    def __init__(self, llm_engine=None):
        self.llm = llm_engine

    def evaluate_priority(self, sender: str, subject: str, content: str, memory_context: str) -> Dict[str, Any]:
        if self.llm:
            prompt = f"Analyze priority (URGENT, HIGH, MEDIUM, LOW) for:\nSender: {sender}\nSubject: {subject}\nContent: {content}\nRespond strictly in JSON: {{\\\"priority_level\\\": \\\"...\\\", \\\"score\\\": 0-100, \\\"reasoning\\\": \\\"...\\\"}}"
            try:
                response = self.llm.invoke(prompt)
                json_match = re.search(r'\{.*\}', response, re.DOTALL)
                if json_match:
                    return json.loads(json_match.group(0))
            except Exception:
                pass
        
        # Fallback baseline prioritization rule
        if "CRITICAL" in subject.upper() or "URGENT" in subject.upper():
            return {"priority_level": "URGENT", "score": 95, "reasoning": "Critical keyword detected in subject line."}
        return {"priority_level": "MEDIUM", "score": 50, "reasoning": "Standard incoming communication."}

class SmartReplyAgent:
    def __init__(self, llm_engine=None):
        self.llm = llm_engine

    def generate_reply(self, sender: str, content: str, memory_context: List[str]) -> str:
        if self.llm:
            prompt = f"Generate a polite and concise professional response to:\nSender: {sender}\nMessage: {content}\nResponse:"
            try:
                return self.llm.invoke(prompt).strip()
            except Exception:
                pass
        return f"Hello {sender.split('@')[0].capitalize()},\n\nThank you for your message regarding '{content[:30]}...'. I have received it and will review it shortly.\n\nBest regards,"

class ApprovalDecisionAgent:
    @staticmethod
    def evaluate(priority: str, risk_status: str, confidence_score: float) -> Dict[str, Any]:
        if risk_status != "SAFE":
            return {"action": "NEVER_AUTO_REPLY", "reason": f"Security risk flag active ({risk_status})."}
        if priority == "URGENT":
            return {"action": "REQUIRES_APPROVAL", "reason": "High priority item requiring human verification."}
        if confidence_score >= 0.85:
            return {"action": "AUTO_REPLY", "reason": "High confidence with zero security risk."}
        return {"action": "REQUIRES_APPROVAL", "reason": "Moderate confidence score."}

class ThreadSummarizationAgent:
    def __init__(self, llm_engine=None):
        self.llm = llm_engine

    def process(self, content: str) -> Dict[str, Any]:
        return {
            "summary": content[:120] + "...",
            "action_items": ["Review content payload"],
            "deadlines": [],
            "meetings": []
        }

class ExplainabilityAgent:
    @staticmethod
    def generate_trace(msg_id: str, priority_res: dict, sec_res: dict, approval_res: dict) -> str:
        return f"""
[TRACE - {msg_id}]
├── Priority : {priority_res.get('priority_level')} (Score: {priority_res.get('score')})
│   └── Reason: {priority_res.get('reasoning')}
├── Security : {sec_res.get('status')} (Risk Score: {sec_res.get('risk_score')})
│   └── Threat Analysis: {sec_res.get('threats_detected')}
└── Decision : {approval_res.get('action')}
    └── Policy Reason: {approval_res.get('reason')}
"""

class MasterAutonomousAgentLoop:
    def __init__(self, llm_engine=None, vector_store=None):
        self.llm = llm_engine
        self.memory = LongTermMemoryAgent(vector_store)
        self.sec_agent = SecurityAgent()
        self.prioritizer = PrioritizationAgent(llm_engine)
        self.reply_agent = SmartReplyAgent(llm_engine)
        self.summarizer = ThreadSummarizationAgent(llm_engine)

    def process_incoming_communication(self, source: str, sender: str, subject: str, content: str) -> Dict[str, Any]:
        event = NotificationAggregationAgent.normalize_event(source, sender, subject, content, "2026-07-21")
        sec_result = self.sec_agent.analyze_message(sender, content)
        past_context = self.memory.query_context(sender, content)
        prio_result = self.prioritizer.evaluate_priority(sender, subject, content, str(past_context))
        summary_result = self.summarizer.process(content)
        draft_reply = self.reply_agent.generate_reply(sender, content, past_context)
        
        approval_result = ApprovalDecisionAgent.evaluate(
            prio_result.get("priority_level"),
            sec_result.get("status"),
            confidence_score=0.90 if sec_result.get("is_safe") else 0.20
        )
        
        trace = ExplainabilityAgent.generate_trace(event["id"], prio_result, sec_result, approval_result)
        
        self._log_to_sqlite(source, sender, subject, prio_result.get("priority_level"), sec_result.get("status"), approval_result.get("action"), trace)

        return {
            "event": event,
            "security": sec_result,
            "priority": prio_result,
            "summary": summary_result,
            "draft_reply": draft_reply,
            "decision": approval_result,
            "trace": trace
        }

    def _log_to_sqlite(self, source, sender, subject, priority, security, decision, reasoning):
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS audit_logs (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                source TEXT, sender TEXT, subject TEXT, priority TEXT,
                security_status TEXT, decision TEXT, reasoning TEXT, raw_payload TEXT
            )
        ''')
        cursor.execute('''
            INSERT INTO audit_logs (source, sender, subject, priority, security_status, decision, reasoning, raw_payload)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (source, sender, subject, priority, security, decision, reasoning, ""))
        conn.commit()
        conn.close()

# ------------------------------------------------------------------------------
# Instantiate Agent Loop
# ------------------------------------------------------------------------------

llm_inst = globals().get('llm', None)
vdb_inst = globals().get('vector_db', None)

agent_loop = MasterAutonomousAgentLoop(llm_inst, vdb_inst)

# ------------------------------------------------------------------------------
# Live Execution Batch
# ------------------------------------------------------------------------------

test_payloads = [
    {
        "source": "Gmail",
        "sender": "engineering-lead@company.com",
        "subject": "CRITICAL: AMD GPU Cluster Outage in Region 2",
        "content": "Our ROCm training jobs are failing due to a driver mismatch on node-04. Can you inspect the logs immediately?"
    },
    {
        "source": "Slack",
        "sender": "security-alert@suspicious-domain-phish.net",
        "subject": "Urgent Security Verification",
        "content": "Your password will expire in 1 hour. Verify password immediately by clicking http://192.168.1.1/login"
    },
    {
        "source": "LinkedIn",
        "sender": "recruiter@techexecs.com",
        "subject": "Senior AI Architect Opportunity",
        "content": "Hi! We loved your profile and work with LLM multi-agent systems. Would you be open for a quick intro call this Thursday at 2 PM IST?"
    }
]

print("================================================================================")
print("              STARTING AUTONOMOUS AGENT TRIAGE SIMULATION                       ")
print("================================================================================")

results = []
for idx, item in enumerate(test_payloads, 1):
    print(f"\n[>>> PROCESSING COMMUNICATION {idx}/{len(test_payloads)}: Source={item['source']} <<<]")
    res = agent_loop.process_incoming_communication(
        source=item["source"],
        sender=item["sender"],
        subject=item["subject"],
        content=item["content"]
    )
    results.append(res)
    print(res["trace"])
    print(f"Drafted Smart Reply:\n\"{res['draft_reply']}\"\n")
    print("-" * 80)

print("[✓] Batch processing simulation completed successfully.")

              STARTING AUTONOMOUS AGENT TRIAGE SIMULATION                       

[>>> PROCESSING COMMUNICATION 1/3: Source=Gmail <<<]

[TRACE - Gmail_3598471432484417925]
├── Priority : URGENT (Score: 95)
│   └── Reason: Critical keyword detected in subject line.
├── Security : SAFE (Risk Score: 0)
│   └── Threat Analysis: []
└── Decision : REQUIRES_APPROVAL
    └── Policy Reason: High priority item requiring human verification.

Drafted Smart Reply:
"Hello Engineering-lead,

Thank you for your message regarding 'Our ROCm training jobs are fai...'. I have received it and will review it shortly.

Best regards,"

--------------------------------------------------------------------------------

[>>> PROCESSING COMMUNICATION 2/3: Source=Slack <<<]

[TRACE - Slack_5165977186723201597]
├── Priority : URGENT (Score: 95)
│   └── Reason: Critical keyword detected in subject line.
├── Security : CRITICAL_PHISHING_RISK (Risk Score: 60)
│   └── Threat Analysis: ['Unverified external domain: @susp

In [1]:
# ==============================================================================
# CELL 11: Multi-Platform Autonomous Simulation & Live Test Execution (FULLY SELF-CONTAINED)
# ==============================================================================

import os
import json
import sqlite3
import re
from typing import List, Dict, Any
from urllib.parse import urlparse

# Ensure DATA_DIR and DB_PATH are set
DATA_DIR = "./agent_data"
DB_PATH = os.path.join(DATA_DIR, "agent_memory.db")
os.makedirs(DATA_DIR, exist_ok=True)

# ------------------------------------------------------------------------------
# Fallback Definitions for All Required Agents (Ensures 100% Independent Execution)
# ------------------------------------------------------------------------------

class SecurityAgent:
    SUSPICIOUS_KEYWORDS = ["urgent wire transfer", "verify password immediately", "account suspended", "gift card", "crypto deposit"]
    KNOWN_DOMAINS = ["company.com", "gmail.com", "github.com", "slack.com", "jira.atlassian.net", "amd.com", "techexecs.com"]

    @classmethod
    def analyze_message(cls, sender: str, body: str, links: List[str] = []) -> Dict[str, Any]:
        risk_score = 0
        reasons = []

        sender_domain = sender.split("@")[-1] if "@" in sender else ""
        if sender_domain and sender_domain not in cls.KNOWN_DOMAINS:
            risk_score += 25
            reasons.append(f"Unverified external domain: @{sender_domain}")

        for kw in cls.SUSPICIOUS_KEYWORDS:
            if kw.lower() in body.lower():
                risk_score += 35
                reasons.append(f"Suspicious social engineering phrase detected: '{kw}'")

        for link in links:
            parsed = urlparse(link)
            if re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', parsed.netloc):
                risk_score += 40
                reasons.append(f"Direct IP raw URL detected: {link}")

        status = "CRITICAL_PHISHING_RISK" if risk_score >= 60 else ("SUSPICIOUS" if risk_score >= 25 else "SAFE")

        return {
            "status": status,
            "risk_score": risk_score,
            "threats_detected": reasons,
            "is_safe": status == "SAFE"
        }

class NotificationAggregationAgent:
    @staticmethod
    def normalize_event(source: str, sender: str, title: str, content: str, timestamp: str) -> Dict[str, Any]:
        return {
            "source": source,
            "sender": sender,
            "title": title,
            "content": content,
            "timestamp": timestamp,
            "id": f"{source}_{abs(hash(content))}"
        }

class LongTermMemoryAgent:
    def __init__(self, vector_store=None):
        self.db = vector_store

    def store_interaction(self, sender: str, summary: str, user_decision: str):
        if self.db:
            from langchain_core.documents import Document
            text_data = f"Sender: {sender} | Summary: {summary} | Action Taken: {user_decision}"
            doc = Document(page_content=text_data, metadata={"sender": sender, "action": user_decision})
            self.db.add_documents([doc])

    def query_context(self, sender: str, query_text: str, k: int = 2) -> List[str]:
        if self.db:
            try:
                results = self.db.similarity_search(f"{sender} {query_text}", k=k)
                return [res.page_content for res in results]
            except Exception:
                pass
        return [f"Prior interaction logged with {sender}"]

class PrioritizationAgent:
    def __init__(self, llm_engine=None):
        self.llm = llm_engine

    def evaluate_priority(self, sender: str, subject: str, content: str, memory_context: str) -> Dict[str, Any]:
        if self.llm:
            prompt = f"Analyze priority (URGENT, HIGH, MEDIUM, LOW) for:\nSender: {sender}\nSubject: {subject}\nContent: {content}\nRespond strictly in JSON: {{\\\"priority_level\\\": \\\"...\\\", \\\"score\\\": 0-100, \\\"reasoning\\\": \\\"...\\\"}}"
            try:
                response = self.llm.invoke(prompt)
                json_match = re.search(r'\{.*\}', response, re.DOTALL)
                if json_match:
                    return json.loads(json_match.group(0))
            except Exception:
                pass
        
        # Fallback baseline prioritization rule
        if "CRITICAL" in subject.upper() or "URGENT" in subject.upper():
            return {"priority_level": "URGENT", "score": 95, "reasoning": "Critical keyword detected in subject line."}
        return {"priority_level": "MEDIUM", "score": 50, "reasoning": "Standard incoming communication."}

class SmartReplyAgent:
    def __init__(self, llm_engine=None):
        self.llm = llm_engine

    def generate_reply(self, sender: str, content: str, memory_context: List[str]) -> str:
        if self.llm:
            prompt = f"Generate a polite and concise professional response to:\nSender: {sender}\nMessage: {content}\nResponse:"
            try:
                return self.llm.invoke(prompt).strip()
            except Exception:
                pass
        return f"Hello {sender.split('@')[0].capitalize()},\n\nThank you for your message regarding '{content[:30]}...'. I have received it and will review it shortly.\n\nBest regards,"

class ApprovalDecisionAgent:
    @staticmethod
    def evaluate(priority: str, risk_status: str, confidence_score: float) -> Dict[str, Any]:
        if risk_status != "SAFE":
            return {"action": "NEVER_AUTO_REPLY", "reason": f"Security risk flag active ({risk_status})."}
        if priority == "URGENT":
            return {"action": "REQUIRES_APPROVAL", "reason": "High priority item requiring human verification."}
        if confidence_score >= 0.85:
            return {"action": "AUTO_REPLY", "reason": "High confidence with zero security risk."}
        return {"action": "REQUIRES_APPROVAL", "reason": "Moderate confidence score."}

class ThreadSummarizationAgent:
    def __init__(self, llm_engine=None):
        self.llm = llm_engine

    def process(self, content: str) -> Dict[str, Any]:
        return {
            "summary": content[:120] + "...",
            "action_items": ["Review content payload"],
            "deadlines": [],
            "meetings": []
        }

class ExplainabilityAgent:
    @staticmethod
    def generate_trace(msg_id: str, priority_res: dict, sec_res: dict, approval_res: dict) -> str:
        return f"""
[TRACE - {msg_id}]
├── Priority : {priority_res.get('priority_level')} (Score: {priority_res.get('score')})
│   └── Reason: {priority_res.get('reasoning')}
├── Security : {sec_res.get('status')} (Risk Score: {sec_res.get('risk_score')})
│   └── Threat Analysis: {sec_res.get('threats_detected')}
└── Decision : {approval_res.get('action')}
    └── Policy Reason: {approval_res.get('reason')}
"""

class MasterAutonomousAgentLoop:
    def __init__(self, llm_engine=None, vector_store=None):
        self.llm = llm_engine
        self.memory = LongTermMemoryAgent(vector_store)
        self.sec_agent = SecurityAgent()
        self.prioritizer = PrioritizationAgent(llm_engine)
        self.reply_agent = SmartReplyAgent(llm_engine)
        self.summarizer = ThreadSummarizationAgent(llm_engine)

    def process_incoming_communication(self, source: str, sender: str, subject: str, content: str) -> Dict[str, Any]:
        event = NotificationAggregationAgent.normalize_event(source, sender, subject, content, "2026-07-21")
        sec_result = self.sec_agent.analyze_message(sender, content)
        past_context = self.memory.query_context(sender, content)
        prio_result = self.prioritizer.evaluate_priority(sender, subject, content, str(past_context))
        summary_result = self.summarizer.process(content)
        draft_reply = self.reply_agent.generate_reply(sender, content, past_context)
        
        approval_result = ApprovalDecisionAgent.evaluate(
            prio_result.get("priority_level"),
            sec_result.get("status"),
            confidence_score=0.90 if sec_result.get("is_safe") else 0.20
        )
        
        trace = ExplainabilityAgent.generate_trace(event["id"], prio_result, sec_result, approval_result)
        
        self._log_to_sqlite(source, sender, subject, prio_result.get("priority_level"), sec_result.get("status"), approval_result.get("action"), trace)

        return {
            "event": event,
            "security": sec_result,
            "priority": prio_result,
            "summary": summary_result,
            "draft_reply": draft_reply,
            "decision": approval_result,
            "trace": trace
        }

    def _log_to_sqlite(self, source, sender, subject, priority, security, decision, reasoning):
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS audit_logs (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                source TEXT, sender TEXT, subject TEXT, priority TEXT,
                security_status TEXT, decision TEXT, reasoning TEXT, raw_payload TEXT
            )
        ''')
        cursor.execute('''
            INSERT INTO audit_logs (source, sender, subject, priority, security_status, decision, reasoning, raw_payload)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (source, sender, subject, priority, security, decision, reasoning, ""))
        conn.commit()
        conn.close()

# ------------------------------------------------------------------------------
# Instantiate Agent Loop
# ------------------------------------------------------------------------------

llm_inst = globals().get('llm', None)
vdb_inst = globals().get('vector_db', None)

agent_loop = MasterAutonomousAgentLoop(llm_inst, vdb_inst)

# ------------------------------------------------------------------------------
# Live Execution Batch
# ------------------------------------------------------------------------------

test_payloads = [
    {
        "source": "Gmail",
        "sender": "engineering-lead@company.com",
        "subject": "CRITICAL: AMD GPU Cluster Outage in Region 2",
        "content": "Our ROCm training jobs are failing due to a driver mismatch on node-04. Can you inspect the logs immediately?"
    },
    {
        "source": "Slack",
        "sender": "security-alert@suspicious-domain-phish.net",
        "subject": "Urgent Security Verification",
        "content": "Your password will expire in 1 hour. Verify password immediately by clicking http://192.168.1.1/login"
    },
    {
        "source": "LinkedIn",
        "sender": "recruiter@techexecs.com",
        "subject": "Senior AI Architect Opportunity",
        "content": "Hi! We loved your profile and work with LLM multi-agent systems. Would you be open for a quick intro call this Thursday at 2 PM IST?"
    }
]

print("================================================================================")
print("              STARTING AUTONOMOUS AGENT TRIAGE SIMULATION                       ")
print("================================================================================")

results = []
for idx, item in enumerate(test_payloads, 1):
    print(f"\n[>>> PROCESSING COMMUNICATION {idx}/{len(test_payloads)}: Source={item['source']} <<<]")
    res = agent_loop.process_incoming_communication(
        source=item["source"],
        sender=item["sender"],
        subject=item["subject"],
        content=item["content"]
    )
    results.append(res)
    print(res["trace"])
    print(f"Drafted Smart Reply:\n\"{res['draft_reply']}\"\n")
    print("-" * 80)

print("[✓] Batch processing simulation completed successfully.")

              STARTING AUTONOMOUS AGENT TRIAGE SIMULATION                       

[>>> PROCESSING COMMUNICATION 1/3: Source=Gmail <<<]

[TRACE - Gmail_8055726997811790549]
├── Priority : URGENT (Score: 95)
│   └── Reason: Critical keyword detected in subject line.
├── Security : SAFE (Risk Score: 0)
│   └── Threat Analysis: []
└── Decision : REQUIRES_APPROVAL
    └── Policy Reason: High priority item requiring human verification.

Drafted Smart Reply:
"Hello Engineering-lead,

Thank you for your message regarding 'Our ROCm training jobs are fai...'. I have received it and will review it shortly.

Best regards,"

--------------------------------------------------------------------------------

[>>> PROCESSING COMMUNICATION 2/3: Source=Slack <<<]

[TRACE - Slack_4887269643367967536]
├── Priority : URGENT (Score: 95)
│   └── Reason: Critical keyword detected in subject line.
├── Security : CRITICAL_PHISHING_RISK (Risk Score: 60)
│   └── Threat Analysis: ['Unverified external domain: @susp

In [16]:
# ==============================================================================
# CELL 12: Interactive Control Dashboard (IPython HTML Display for Cloud Workspaces)
# ==============================================================================

import os
import sqlite3
import pandas as pd
from IPython.display import display, HTML, clear_output

def fetch_audit_logs():
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql_query("SELECT id, timestamp, source, sender, priority, security_status, decision FROM audit_logs ORDER BY id DESC LIMIT 5", conn)
        conn.close()
        return df
    except Exception:
        return pd.DataFrame()

def render_dashboard():
    """Renders a pure native HTML/JS UI that works inside JupyterLab cloud proxies without Gradio network blockages."""
    
    # Run a test execution through the agent loop to display live data
    test_res = agent_loop.process_incoming_communication(
        source="Gmail",
        sender="security-lead@amd.com",
        subject="ROCm 6.1 Cluster Optimization Verification",
        content="The local multi-agent inference benchmarking on AMD Ryzen AI hardware is complete. Throughput increased by 42%. Approval requested for production push."
    )
    
    logs_df = fetch_audit_logs()
    logs_html = logs_df.to_html(classes="table table-striped", index=False) if not logs_df.empty else "<p>No log records found.</p>"

    html_content = f"""
    <div style="font-family: Arial, sans-serif; background-color: #0f172a; color: #f8fafc; padding: 20px; border-radius: 12px; border: 1px solid #334155;">
        <h2 style="color: #38bdf8; margin-top: 0;">🚀 AMD AI Autonomous Email & Notification Triage Dashboard</h2>
        <p style="color: #94a3b8; font-size: 14px;">Running on AMD Ryzen™ AI / ROCm Acceleration Engine | Powered by Ollama & LangChain</p>
        <hr style="border-color: #334155;" />
        
        <div style="display: flex; gap: 20px; margin-bottom: 20px;">
            <div style="flex: 1; background: #1e293b; padding: 15px; border-radius: 8px; border: 1px solid #475569;">
                <h3 style="color: #f1f5f9; margin-top:0;">📥 Active Event Ingestion</h3>
                <p><b>Source:</b> <span style="color: #a855f7;">{test_res['event']['source']}</span></p>
                <p><b>Sender:</b> {test_res['event']['sender']}</p>
                <p><b>Subject:</b> {test_res['event']['title']}</p>
                <div style="background: #0f172a; padding: 10px; border-radius: 6px; font-size: 13px; color: #cbd5e1;">
                    {test_res['event']['content']}
                </div>
            </div>
            
            <div style="flex: 1; background: #1e293b; padding: 15px; border-radius: 8px; border: 1px solid #475569;">
                <h3 style="color: #f1f5f9; margin-top:0;">🧠 Autonomous Agent Evaluation</h3>
                <p><b>Priority:</b> <span style="color: #ef4444; font-weight: bold;">{test_res['priority'].get('priority_level', 'MEDIUM')}</span> (Score: {test_res['priority'].get('score', 50)})</p>
                <p><b>Security Status:</b> <span style="color: #22c55e; font-weight: bold;">{test_res['security'].get('status', 'SAFE')}</span></p>
                <p><b>Autonomous Action:</b> <span style="color: #3b82f6; font-weight: bold;">{test_res['decision'].get('action', 'AUTO_REPLY')}</span></p>
                <p><b>Generated Summary:</b> {test_res['summary'].get('summary', 'N/A')}</p>
            </div>
        </div>
        
        <div style="background: #1e293b; padding: 15px; border-radius: 8px; margin-bottom: 20px; border: 1px solid #475569;">
            <h3 style="color: #f1f5f9; margin-top:0;">💬 Smart Generated Response</h3>
            <pre style="background: #0f172a; color: #a3e635; padding: 12px; border-radius: 6px; white-space: pre-wrap; font-family: monospace;">{test_res['draft_reply']}</pre>
        </div>

        <div style="background: #1e293b; padding: 15px; border-radius: 8px; margin-bottom: 20px; border: 1px solid #475569;">
            <h3 style="color: #f1f5f9; margin-top:0;">🔍 Explainability Decision Trace</h3>
            <pre style="background: #0f172a; color: #38bdf8; padding: 12px; border-radius: 6px; white-space: pre-wrap; font-family: monospace;">{test_res['trace']}</pre>
        </div>

        <div style="background: #1e293b; padding: 15px; border-radius: 8px; border: 1px solid #475569;">
            <h3 style="color: #f1f5f9; margin-top:0;">📊 SQLite Decision Audit Logs</h3>
            <div style="overflow-x: auto; color: #e2e8f0;">
                {logs_html}
            </div>
        </div>
    </div>
    """
    display(HTML(html_content))

render_dashboard()

id,timestamp,source,sender,priority,security_status,decision
19,2026-07-23 11:46:54,Gmail,security-lead@amd.com,MEDIUM,SAFE,AUTO_REPLY
18,2026-07-23 11:46:51,LinkedIn,recruiter@techexecs.com,MEDIUM,SAFE,AUTO_REPLY
17,2026-07-23 11:46:51,Slack,security-alert@suspicious-domain-phish.net,URGENT,CRITICAL_PHISHING_RISK,NEVER_AUTO_REPLY
16,2026-07-23 11:46:51,Gmail,engineering-lead@company.com,URGENT,SAFE,REQUIRES_APPROVAL
15,2026-07-23 11:43:20,LinkedIn,recruiter@techexecs.com,MEDIUM,SAFE,AUTO_REPLY
